# ADaM trials — mapping to the common schema

**Trials covered:** `NCT00113763` (BSC — panitumumab + best supportive care vs. BSC alone) and
`NCT00364013_ADaM` (the ADaM-format copy of PRIME, same trial as the raw-domain `NCT00364013_raw`
already harmonized). These two ship **only pre-derived ADaM domains** (`adsl`, `adae`, `adlb`,
`adls`, `adpm`, `adrsp`, `biomark`), not raw SDTM-like domains — no `disposit`/`demo`/`ae`/`death`
files to map by direct analogy, so this needs its own investigation.

**Headline correction to the earlier harmonization work:** while mapping ADaM's `adsl`, found that
`PFSCR`/`PFSDYCR` — "PD on Study (Central, RECIST) **or Death**", the composite progression-free-
survival endpoint — exists **identically** (same field name, same label) in the raw domains'
`a_eendpt` too, alongside the pure-progression `PDCR`/`PDDYCR` recommended in
`dropout_reason_harmonization.ipynb`. Missed it there because that search only matched column
names containing `"PD"`/`"PROG"` — `PFSCR` doesn't contain either substring. Verified it behaves
identically (event=1/0 flag, day populated for every patient) across all 3 raw trials too.
**`PFSCR`/`PFSDYCR` is the better harmonization target** — present with identical semantics in
literally all 5 trial exports (3 raw + 2 ADaM), not just the 3 raw ones.

**Other findings, by domain:**
1. **Discontinuation reason is genuinely absent from both ADaM trials** — confirmed by keyword
   search (`REAS`, `DISC`, `STOP`, `WD`, `WITHDR`, `DISPO`, `EOIP`, `EOT`) across every column
   name/label in every ADaM domain file. Real methodological limitation: these 2 trials can
   contribute to composite PFS/OS survival modeling (Cox/RSF), but **not** to the admin-reason-
   driven modifiable-driver analysis (§5.3/§6 of the design spec) — that needs `EOIP`/`DSREAS`,
   which only the 3 raw-domain trials have.
2. **`adsl` demographics mostly map cleanly** (`AGE`, `SEX`, `B_ECOG`, `DIAGTYPE`) but `RACE` is
   **more granular** than the raw domains' `RACCAT` (5 categories incl. Hispanic/Asian vs. 3
   collapsed buckets) — needs re-bucketing to the coarser scheme for consistency across all 5
   trials, not a straight rename.
3. **`TRT` vs `ATRT`** (assigned vs. actual treatment) — disagree for exactly 1 patient in each of
   these 2 trials (checked below, not eyeballed from margins). Rare, but a per-protocol vs.
   intent-to-treat modeling choice to make explicitly, not silently default to one.
4. **`B_HEIGHT` is missing entirely from BSC's `adsl`** — BSA can't be derived for that trial from
   this domain (FOLFOX_ADaM has both weight and height).
5. **`adae` has a thinner field set than the raw domains** — severity (`AESEVCD`, confirmed same
   1-5 scale) is there, but `AEREL` (relatedness), `AESER` (seriousness), and `MEDDRA_V` are
   **not** — AE severity burden can be modeled for these 2 trials, but not relatedness/seriousness.
6. **Biomarker mapping already solved** (from `demo_harmonization.ipynb`) — same `BMMTNM*`/`BMMTR*`
   name/value structure in both ADaM trials' `biomark` file. One wrinkle: BSC's `biomark` only has
   7 exon slots (no extended RAS testing) vs. FOLFOX_ADaM's 9 — a real data-availability
   difference (extended RAS panels became standard later), not an error.

In [ ]:
import pyreadstat
from pathlib import Path
import pandas as pd

pd.set_option('display.max_rows', None)

DATA_DIR = Path('..') / '..' / 'Data' / 'raw'

ADAM_TRIALS = {
    'BSC (NCT00113763)': 'NCT00113763_panitumumab_bsc_ADaM',
    'FOLFOX/PRIME (NCT00364013_ADaM)': 'NCT00364013_panitumumab_folfox_ADaM',
}
RAW_TRIALS = {
    'PACCE (NCT00115225)': 'NCT00115225_PACCE_bev_panitumumab_raw',
    'FOLFIRI (NCT00339183)': 'NCT00339183_panitumumab_folfiri_raw',
    'FOLFOX/PRIME (NCT00364013_raw)': 'NCT00364013_panitumumab_folfox_raw',
}
ADAM_DOMAINS = ['adsl_pds2019.sas7bdat', 'adae_pds2019.sas7bdat', 'adlb_pds2019.sas7bdat',
                'adls_pds2019.sas7bdat', 'adpm_pds2019.sas7bdat', 'adrsp_pds2019.sas7bdat',
                'biomark_pds2019.sas7bdat']

## 0. Full ADaM domain map — every column, every domain, both trials

Ground truth before mapping anything — the raw domains have dozens of columns per file; these
ADaM domains are much thinner (already-derived, analysis-ready), so a full column dump is
actually readable here, unlike the raw-domain survey.

In [ ]:
for label, folder in ADAM_TRIALS.items():
    print(f'########## {label} ##########')
    for dom in ADAM_DOMAINS:
        _, meta = pyreadstat.read_sas7bdat(str(DATA_DIR / folder / dom), metadataonly=True)
        print(f'--- {dom} ---')
        for c, l in zip(meta.column_names, meta.column_labels):
            print(f'  {c:12s} -> {l}')
        print()
    print()

## 1. Discontinuation reason — confirm it's genuinely absent (not another `PDYN`-style miss)

Search every column name AND label across every ADaM domain file for reason/discontinuation-like
keywords, before concluding the field just doesn't exist.

In [ ]:
keywords = ['REAS', 'DISC', 'STOP', 'WD', 'WITHDR', 'DISPO', 'EOIP', 'EOT']
hits = []
for label, folder in ADAM_TRIALS.items():
    for dom in ADAM_DOMAINS:
        _, meta = pyreadstat.read_sas7bdat(str(DATA_DIR / folder / dom), metadataonly=True)
        for c, l in zip(meta.column_names, meta.column_labels):
            if any(k in c.upper() for k in keywords) or any(k in (l or '').upper() for k in keywords):
                hits.append((label, dom, c, l))

print(f'{len(hits)} matches found (expect 0):')
for h in hits:
    print(h)

## 2. `PFSCR`/`PFSDYCR` — the corrected, universal harmonization target

**What it means:** `PFSCR` = "PD on Study (Central, RECIST) or Death" — a 1/0 composite endpoint
flag: progressed OR died, whichever came first (the standard oncology "progression-free survival"
event definition). `PFSDYCR` = the paired day (event day if `PFSCR==1`, censoring day otherwise —
same event/censoring-day duality as `PDCR`/`PDDYCR` from the earlier investigation).

Confirmed below: present with identical field name + label in ADaM's `adsl` (patient-level, one
row per subject) AND in the raw domains' `a_eendpt` (already checked in
`dropout_reason_harmonization.ipynb`, re-verified here) — same 1/0 coding, same populated-for-
everyone day field, across **all 5 trial exports**.

In [ ]:
print('=== ADaM trials — adsl.PFSCR/PFSDYCR ===')
for label, folder in ADAM_TRIALS.items():
    df, meta = pyreadstat.read_sas7bdat(str(DATA_DIR / folder / 'adsl_pds2019.sas7bdat'))
    label_lookup = dict(zip(meta.column_names, meta.column_labels))
    print(f'{label} (n={len(df)}) — PFSCR label: {label_lookup["PFSCR"]}')
    print(df['PFSCR'].value_counts(dropna=False).to_dict())
    print('PFSDYCR non-null:', df['PFSDYCR'].notna().sum(), '/', len(df))
    print()

print('=== Raw-domain trials — a_eendpt.PFSCR/PFSDYCR (re-verification) ===')
for label, folder in RAW_TRIALS.items():
    df, meta = pyreadstat.read_sas7bdat(str(DATA_DIR / folder / 'a_eendpt.sas7bdat'))
    label_lookup = dict(zip(meta.column_names, meta.column_labels))
    print(f'{label} (n={len(df)}) — PFSCR label: {label_lookup["PFSCR"]}')
    print(df['PFSCR'].value_counts(dropna=False).to_dict())
    print('PFSDYCR non-null:', df['PFSDYCR'].notna().sum(), '/', len(df))
    print()

## 3. `adsl` demographics — value comparison against the raw-domain `demo` conventions

**What's new/different vs. `demo_harmonization.ipynb`:**
- **`RACE`** — more granular than raw `RACCAT` (adds Hispanic/Latino, Asian as distinct
  categories instead of collapsing into "Other"). Needs re-bucketing down to the raw domains'
  3-category scheme (White/Black/Other) for a consistent column across all 5 trials, since the
  raw trials can't be un-collapsed the other way.
- **`TRT` vs `ATRT`** — assigned vs. actual treatment. Where they disagree, a patient didn't
  receive what they were randomized to (crossover, protocol deviation) — checked below, exactly
  1 patient per trial. Which one becomes the harmonized `on_panitumumab` flag is still an
  ITT-vs-per-protocol modeling choice to make explicitly, even though the practical impact here
  is tiny.
- **`B_ECOG`** — same category labels as raw `B_ECOGI`, but BSC's population reaches grade 3
  ("In bed more than 50% of the time") which didn't appear in the raw trials' samples — expected,
  since BSC is a sicker, treatment-refractory population (best-supportive-care-only arm exists).
- **`B_HEIGHT`** — present for FOLFOX_ADaM, **absent entirely** from BSC's `adsl` — BSA can't be
  computed for BSC patients from this domain.

In [ ]:
adsl_cols = ['TRT', 'ATRT', 'AGE', 'SEX', 'RACE', 'B_ECOG', 'DIAGTYPE', 'B_WEIGHT', 'B_HEIGHT']

for label, folder in ADAM_TRIALS.items():
    df, meta = pyreadstat.read_sas7bdat(str(DATA_DIR / folder / 'adsl_pds2019.sas7bdat'))
    label_lookup = dict(zip(meta.column_names, meta.column_labels))
    print(f'===== {label} — adsl, n={len(df)} =====')
    for col in adsl_cols:
        if col not in df.columns:
            print(f'-- {col}: NOT PRESENT')
            continue
        lbl = label_lookup.get(col, '')
        if df[col].nunique(dropna=False) <= 12:
            print(f'-- {col} ({lbl}):')
            print(df[col].value_counts(dropna=False))
        else:
            print(f'-- {col} ({lbl}): numeric, describe:')
            print(df[col].describe())
        print()
    print()

In [ ]:
# TRT vs ATRT disagreement count — how many patients didn't get what they were assigned
for label, folder in ADAM_TRIALS.items():
    df, meta = pyreadstat.read_sas7bdat(str(DATA_DIR / folder / 'adsl_pds2019.sas7bdat'))
    disagree = (df['TRT'] != df['ATRT']).sum()
    print(f'{label}: {disagree} / {len(df)} patients where TRT != ATRT')

## 4. `adae` — severity scale check + confirm the missing fields

**What's confirmed:** `AESEVCD` uses the same 1-5 grade scale as the raw domains' `AESEVCD`
(checked the actual coded values below, not just the presence of the column).

**What's missing** (already surfaced in section 0's full column dump, called out explicitly
here): no `AEREL` (relatedness), no `AESER` (seriousness), no `MEDDRA_V`. AE severity burden is
still usable as a model feature for these 2 trials; relatedness/seriousness-based features
are not.

In [ ]:
for label, folder in ADAM_TRIALS.items():
    df, meta = pyreadstat.read_sas7bdat(str(DATA_DIR / folder / 'adae_pds2019.sas7bdat'))
    label_lookup = dict(zip(meta.column_names, meta.column_labels))
    print(f'===== {label} — adae, n={len(df)} rows =====')
    print(f"AESEVCD label: {label_lookup['AESEVCD']}")
    print(df['AESEVCD'].value_counts(dropna=False).sort_index())
    print()
    for missing_col in ['AEREL', 'AESER', 'MEDDRA_V']:
        print(f'-- {missing_col}: {"present" if missing_col in df.columns else "NOT PRESENT"}')
    print()

## 5. `biomark` — confirm the exon-slot count difference

Already solved the name/value structure in `demo_harmonization.ipynb` for FOLFOX_ADaM. Confirming
here that BSC's `biomark` file has fewer exon slots — a real data-availability difference (BSC's
biomarker panel predates the extended RAS testing added later), not a parsing error.

In [ ]:
for label, folder in ADAM_TRIALS.items():
    _, meta = pyreadstat.read_sas7bdat(str(DATA_DIR / folder / 'biomark_pds2019.sas7bdat'), metadataonly=True)
    slots = [c for c in meta.column_names if c.startswith('BMMTNM')]
    print(f'{label}: {len(slots)} exon slots -> {slots}')

## Summary — harmonized schema coverage across all 5 trial exports

| Field | PACCE | FOLFIRI | FOLFOX/PRIME (raw) | BSC (ADaM) | FOLFOX/PRIME (ADaM) |
|---|---|---|---|---|---|
| `trial_id` / `SUBJID` | Y | Y | Y | Y | Y |
| Progression-or-death (`PFSCR`/`PFSDYCR`) | Y | Y | Y | Y | Y |
| Discontinuation reason (`EOIP`/`DSREAS`) | Y | Y | Y | **N** | **N** |
| Death (`DTHDY`) | Y | Y | Y | Y (`DTHDYX`) | Y (`DTHDY`) |
| Demographics (age/sex/race/ECOG/diagnosis) | Y | Y | Y | Y | Y |
| Treatment arm (needs derived mapping) | Y | Y | Y | Y | Y |
| KRAS (needs cross-file join for 2 trials) | Y | Y | join | join | join |
| Height/BSA | Y | Y | Y | **N** | Y |
| AE severity | Y | Y | Y | Y | Y |
| AE relatedness/seriousness/MedDRA version | Y | Y | Y | **N** | **N** |

The 2 ADaM trials are usable for the full Cox/RSF survival modeling stages (§5.1-5.2) on the
universal `PFSCR`/`PFSDYCR` composite endpoint, but **not** for the modifiable-driver TMLE/agent
stages (§5.3/§6) that depend on discontinuation reason — that's a real scope limitation to state
explicitly in the writeup, not smooth over.